# 12 · Token probabilities of the first tagger, and a second tagger

Notebook 11 summarized the tagger's token probabilities with a smoothing window of 32 tokens. The remaining fusion
errors are texts where the AI rewrote only one or two sentences (median 2.6% of the words), about 15 tokens: averaged
over 32 tokens, such a section loses half of its signal. Notebook 10 kept only the eight per-text summaries, so this
notebook does two things on the GPU:

1. **First tagger, token probabilities.** The tagger of notebook 10 (attached as an input, no retraining) predicts every
   token of the validation and test texts again, and the probabilities are saved with the character position of every
   token. Finer summaries (shorter windows, sentence means) are then computed on the Mac in notebook 13.
2. **Second tagger.** The same recipe as notebook 10, trained on more data with another seed: all 68,679 training-part
   fusion texts with recovered AI sections (instead of 50,000), 40,000 human and 25,000 ai texts. Both taggers use the
   same tokenizer, so their token probabilities can be averaged token by token in notebook 13.

The shared code (chunks, labels, prediction, files) is in `src/hwkit/tagger.py`.

In [1]:
# Setup: clone/update the repo on Colab/Kaggle, install requirements, resolve paths, get the data
import os, sys, subprocess
REPO_URL = "https://github.com/anna-kryvulia/ucu-ai-hw-generated-text-detection.git"
BRANCH = "main"   # branch with the code you pushed
if "google.colab" in sys.modules or os.path.exists("/kaggle/working"):
    ROOT = "/content/repo" if "google.colab" in sys.modules else "/kaggle/working/repo"
    if not os.path.exists(ROOT):
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, ROOT], check=True)
    else:
        subprocess.run(["git", "-C", ROOT, "pull", "-q"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{ROOT}/requirements.txt"], check=True)
else:
    ROOT = os.path.abspath("..")
sys.path.insert(0, f"{ROOT}/src")

from hwkit import env
from hwkit.download import ensure_data
from hwkit.tracker import Run
P = env.setup(seed=42)
ensure_data(P)

restored run tagger_xlmr from /kaggle/input/notebooks/annakryvulia/notebook8b3125bf0a/outputs/runs
platform: kaggle
    data: /kaggle/input/competitions/ai-generated-text-detection-hw3-ucu-ai-2026
 outputs: /kaggle/working/outputs
   cache: /tmp/hwkit_cache
    repo: /kaggle/working/repo
  device: cuda
data ready at /kaggle/input/competitions/ai-generated-text-detection-hw3-ucu-ai-2026


PosixPath('/kaggle/input/competitions/ai-generated-text-detection-hw3-ucu-ai-2026')

In [2]:
# Imports, config; training sample of the second tagger, validation, test; the first tagger's weights
os.environ["TOKENIZERS_PARALLELISM"] = "false"     # the tokenizer runs before DataLoader workers are forked
import time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from transformers import AutoModelForTokenClassification, AutoTokenizer, get_linear_schedule_with_warmup
from transformers.utils import logging as hf_logging
from huggingface_hub.utils import logging as hub_logging

from hwkit.config import load_config
from hwkit.split import LABELS, load_split
from hwkit.tagger import IGNORE, Chunks, RandomChunk, load_tokens, save_tokens, token_probs, token_scores, tokenize

# library info messages (the token classification layer is new and untrained, download hints); errors still show
hf_logging.set_verbosity_error()
hub_logging.set_verbosity_error()
assert torch.cuda.is_available(), "turn on the GPU (Kaggle: Settings -> Accelerator -> GPU T4 or P100)"
DEV = "cuda"
NOTEBOOK_START = time.time()
cfg = load_config("tagger_xlmr2")
TC, DC, SEED = cfg["train"], cfg["data"], cfg["seed"]

first = cfg["inputs"]["first_tagger"]
found = sorted(Path("/kaggle/input").glob(f"**/{first}/checkpoints/model.pt")) + [P.runs / first / "checkpoints" / "model.pt"]
FIRST_MODEL = next((f for f in found if f.exists()), None)
assert FIRST_MODEL is not None, "attach the output of notebook 10 (Kaggle: Add Input -> Your Work -> its notebook)"

train = load_split(pd.read_csv(P.data / "train.csv"))
span_of = pd.read_csv(P.repo / "results" / "fusion_spans.csv.gz").set_index("id")["spans"]   # NaN: no AI section found
part = train[train["fold"] != 0]
fusion = part[(part["label"] == "fusion") & part["id"].isin(span_of.dropna().index)]
fit = pd.concat([fusion if DC["n_fusion"] is None else fusion.sample(n=DC["n_fusion"], random_state=SEED),
                 part[part["label"] == "human"].sample(n=DC["n_human"], random_state=SEED),
                 part[part["label"] == "ai"].sample(n=DC["n_ai"], random_state=SEED)]).reset_index(drop=True)
val = train[train["fold"] == 0].reset_index(drop=True)
test = pd.read_csv(P.data / "test.csv")
del train, part, fusion
monitor_idx = np.random.default_rng(42).choice(len(val), TC["monitor_size"], replace=False)   # same as notebook 10
run = Run(cfg["name"], cfg, P)
print(f"first tagger: {FIRST_MODEL}")
print(f"training: {fit['label'].value_counts().to_dict()}; validation: {len(val)}; test: {len(test)}; "
      f"GPU: {torch.cuda.get_device_name()}")

run 'tagger_xlmr2' at /kaggle/working/outputs/runs/tagger_xlmr2  done: nothing yet
first tagger: /kaggle/input/notebooks/annakryvulia/notebook8b3125bf0a/outputs/runs/tagger_xlmr/checkpoints/model.pt
training: {'fusion': 68679, 'human': 40000, 'ai': 25000}; validation: 27236; test: 25000; GPU: Tesla T4


In [3]:
# Tokenize every text without length limit; token labels for the training sample and validation
tok = AutoTokenizer.from_pretrained(cfg["model"]["arch"])
chunks = Chunks(tok, cfg["model"]["max_len"])
t0 = time.time()
data = {"fit": tokenize(tok, fit, span_of), "val": tokenize(tok, val, span_of), "test": tokenize(tok, test)}
for df in (fit, val, test):
    df.drop(columns="text", inplace=True)              # token ids, labels and positions are all that is needed now
print(f"tokenized in {(time.time() - t0) / 60:.1f} min")
pd.Series([l.mean() for l in data["fit"][1]]).groupby(fit["label"]).describe()[
    ["count", "mean", "25%", "50%", "75%"]].reindex(LABELS).round(3).rename_axis("share of AI tokens")

tokenized in 8.4 min


,count,mean,25%,50%,75%
share of AI tokens,,,,,
human,40000.0,0.000,0.000,0.000,0.000
fusion,68679.0,0.243,0.117,0.226,0.355
ai,25000.0,1.000,1.000,1.000,1.000


## First tagger: token probabilities for validation and test

In [4]:
# The tagger of notebook 10 predicts every token again; probabilities and character positions are saved per text
def tokens_path(name):
    return run.dir / "predictions" / f"{name}.npz"

def new_model():
    return AutoModelForTokenClassification.from_pretrained(
        cfg["model"]["arch"], num_labels=2, id2label={0: "human", 1: "ai"}, label2id={"human": 0, "ai": 1})

def per_class_scores(probs, labels):
    """Token precision/recall/F1 of the AI class on validation, by the true label of the text."""
    rows = {lab: token_scores([probs[i] for i in np.flatnonzero((val["label"] == lab).to_numpy())],
                              [labels[i] for i in np.flatnonzero((val["label"] == lab).to_numpy())]) for lab in LABELS}
    return pd.DataFrame({**rows, "all": token_scores(probs, labels)}).T

probs1 = {}
if not run.is_done("tagger1_tokens"):
    t0 = time.time()
    model = new_model()
    model.load_state_dict({k: v.float() for k, v in torch.load(FIRST_MODEL, map_location="cpu", weights_only=True).items()})
    model.to(DEV)
    for name, df in (("val", val), ("test", test)):
        probs1[name] = token_probs(model, chunks, data[name][0], TC["eval_batch_size"], DEV)
        save_tokens(tokens_path(f"tokens_tagger1_{name}"), df["id"], probs1[name], data[name][2])
    del model
    torch.cuda.empty_cache()
    run.mark_done("tagger1_tokens")
    print(f"{(time.time() - t0) / 60:.1f} min")
else:
    probs1["val"] = load_tokens(tokens_path("tokens_tagger1_val"))[1]
first_scores = per_class_scores(probs1["val"], data["val"][1])
first_scores.round(3)

12.6 min


,token_precision,token_recall,token_f1
human,0.000,0.000,0.000
fusion,0.952,0.976,0.964
ai,1.000,0.999,0.999
all,0.989,0.996,0.992


The first tagger's scores are exactly those of notebook 10 (fusion token F1 0.964), so the saved token probabilities are the same predictions, now kept per token instead of only as per-text summaries.

## Second tagger: fine-tuning

The recipe of notebook 10: AdamW, learning rate 2e-5 with warmup and linear decay, mixed precision, one random
512-token chunk per text and epoch, a checkpoint after every epoch. Only the data and the seed differ.

In [5]:
# Model, optimizer, schedule; resume from the last epoch checkpoint if there is one
torch.manual_seed(SEED)                                    # another initialization of the new token layer
model = new_model().to(DEV)
loader = DataLoader(RandomChunk(chunks, data["fit"][0], data["fit"][1]), batch_size=TC["batch_size"], shuffle=True,
                    drop_last=True, collate_fn=chunks.collate, num_workers=2, pin_memory=True,
                    generator=torch.Generator().manual_seed(SEED))
steps = len(loader) * TC["epochs"]
no_decay = ("bias", "LayerNorm.weight")                    # usual practice: no weight decay on these
params = [{"params": [p for n, p in model.named_parameters() if not n.endswith(no_decay)],
           "weight_decay": TC["weight_decay"]},
          {"params": [p for n, p in model.named_parameters() if n.endswith(no_decay)], "weight_decay": 0.0}]
opt = torch.optim.AdamW(params, lr=TC["lr"])
sched = get_linear_schedule_with_warmup(opt, int(TC["warmup"] * steps), steps)
scaler = torch.amp.GradScaler("cuda")
start_epoch = 0
if not run.is_done("train") and run.ckpt("last.pt").exists():
    state = torch.load(run.ckpt("last.pt"), map_location="cpu", weights_only=False)
    for obj, key in ((model, "model"), (opt, "opt"), (sched, "sched"), (scaler, "scaler")):
        obj.load_state_dict(state[key])
    start_epoch = state["epoch"]
    del state
print(f"{len(loader)} steps per epoch, {TC['epochs']} epochs; starting at epoch {start_epoch + 1}")

8354 steps per epoch, 2 epochs; starting at epoch 1


In [6]:
# Training loop; token-level F1 of the AI class on the monitor sample after each epoch
if not run.is_done("train"):
    mon_ids = [data["val"][0][i] for i in monitor_idx]
    mon_lab = [data["val"][1][i] for i in monitor_idx]
    for epoch in range(start_epoch, TC["epochs"]):
        model.train()
        t0, loss_sum, seen = time.time(), 0.0, 0
        for input_ids, mask, labels in loader:
            input_ids, mask, labels = input_ids.to(DEV), mask.to(DEV), labels.to(DEV)
            with torch.autocast("cuda", dtype=torch.float16):
                logits = model(input_ids=input_ids, attention_mask=mask).logits
            loss = F.cross_entropy(logits.float().flatten(0, 1), labels.flatten(), ignore_index=IGNORE)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scale = scaler.get_scale()
            scaler.step(opt)                      # skipped when float16 gradients overflowed
            scaler.update()
            if scaler.get_scale() >= scale:       # move the schedule only when the optimizer really stepped
                sched.step()
            loss_sum += loss.item() * len(input_ids)
            seen += len(input_ids)
        s = token_scores(token_probs(model, chunks, mon_ids, TC["eval_batch_size"], DEV), mon_lab)
        run.log_epoch(epoch + 1, {"train_loss": loss_sum / seen, **{f"monitor_{k}": v for k, v in s.items()},
                                  "minutes": (time.time() - t0) / 60})
        torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(),
                    "scaler": scaler.state_dict(), "epoch": epoch + 1}, run.ckpt("last.pt"))
        print(f"epoch {epoch + 1}: loss {loss_sum / seen:.4f}, monitor token F1 {s['token_f1']:.4f}, "
              f"{(time.time() - t0) / 60:.1f} min")
    torch.save({k: v.half() for k, v in model.state_dict().items()}, run.ckpt("model.pt"))   # weights only
    run.ckpt("last.pt").unlink()                  # optimizer state is no longer needed (saves 2 GB of output)
    run.mark_done("train")
else:
    model.load_state_dict({k: v.float() for k, v in
                           torch.load(run.ckpt("model.pt"), map_location="cpu", weights_only=True).items()})
run.history.round(4)

epoch 1: loss 0.0617, monitor token F1 0.9939, 71.3 min
epoch 2: loss 0.0228, monitor token F1 0.9943, 71.3 min


,epoch,time,train_loss,monitor_token_precision,monitor_token_recall,monitor_token_f1,minutes
0,1,2026-11-22 13:46:49,0.0617,0.9915,0.9964,0.9939,71.2757
1,2,2026-11-22 14:58:06,0.0228,0.9912,0.9975,0.9943,71.2050


The second tagger sees 133,679 training texts instead of 100,000 (all fusion texts with recovered AI sections, more human and ai texts) and another seed, so an epoch takes 71 minutes instead of 52. Its monitor token F1 after two epochs is 0.994 (first tagger: 0.993).

## Second tagger: token probabilities, and both taggers compared

In [7]:
# Token probabilities of the second tagger for validation and test, saved like the first tagger's
probs2 = {}
if not run.is_done("tagger2_tokens"):
    t0 = time.time()
    for name, df in (("val", val), ("test", test)):
        probs2[name] = token_probs(model, chunks, data[name][0], TC["eval_batch_size"], DEV)
        save_tokens(tokens_path(f"tokens_{name}"), df["id"], probs2[name], data[name][2])
    run.mark_done("tagger2_tokens")
    print(f"{(time.time() - t0) / 60:.1f} min")
else:
    probs2["val"] = load_tokens(tokens_path("tokens_val"))[1]

12.6 min


In [8]:
# Token-level F1 of the AI class on validation: first tagger, second tagger, and their token-wise average
avg = [(a + b) / 2 for a, b in zip(probs1["val"], probs2["val"])]
cmp = pd.concat({"first": first_scores["token_f1"],
                 "second": per_class_scores(probs2["val"], data["val"][1])["token_f1"],
                 "average": per_class_scores(avg, data["val"][1])["token_f1"]}, axis=1)
for col in cmp:
    run.log_metrics(f"val_tokens_{col}", {"token_f1": cmp.loc["all", col], "token_f1_fusion": cmp.loc["fusion", col]})
cmp.round(4)

,first,second,average
human,0.0000,0.0000,0.0000
fusion,0.9641,0.9679,0.9683
ai,0.9993,0.9996,0.9994
all,0.9923,0.9933,0.9934


On validation the second tagger finds AI tokens in fusion texts a little better than the first (F1 0.968 vs 0.964); the token-wise average of both is 0.968. Whether this also helps the text-level decision is tested in notebook 13.

In [9]:
# Distinct warnings logged in this session (should be empty), total time
print(f"total: {(time.time() - NOTEBOOK_START) / 60:.1f} min")
env.warnings_summary(P)

total: 176.8 min


[]